# ALMA-C11 + CEERS analogues — SED comparison, attenuation, torus inclination

Analysis companion of `analogues_specphot_almac11_rt.ipynb`. Reads ONLY through
`almac11_specphot/almac11_source_map.fits` (built by the staging notebook's Part 6 census), so
reused CEERS trees and new `sed_almac11` trees are transparent.

**Sightline convention.** The dust arms (`dust_on`/`dust_off`) carry 4 sightlines
(`THETA=[0,45,90,135]`), the Nenkova arms a single one (`THETA=[0]`, `findx=0`); the reused
CEERS `nenkova_i90` tree carries 4. Whenever arms are differenced per galaxy
(attenuation, torus residuals) the dust arms are read at `findx=0` and multi-sightline Nenkova
SEDs are averaged; for stacks against observations the sightline mean is used (<0.5% scatter
for these dust-poor hosts either way).

**Observed side.** The CSV carries the CIGALE best-model fluxes (`best.<band>`, mJy) for
19 bands (MegaCam/Suprime u->i, VISTA JHKs, IRAC 1-4, MIPS 24, PACS 100, SPIRE 250, ALMA band 6)
— plotted at band pivot wavelengths. Observed photometry with errors is NOT in the CSV; the
`OBS_PHOT_HOOK` below is the place to point at a staged CIGALE `observations.fits` when copied
to the cluster. AGN properties (L_AGN for the torus-rescaling panel) live in
`ALMAC11_target+control_jwst_AGN_prop.fits` — local only as of 2026-08-04; copy to
`obs_data/almac11/` to enable that panel.

**Parts**: 1 observed side · 2 mock stacks per target · 3 attenuation A_lambda / A_V vs CIGALE
Av_ISM · 4 torus MIR vs inclination (i=90/60/30) · 5 dust masses (sanitized CSV column).


In [ ]:
# ── Part 0 · config + resolvers ──────────────────────────────────────────────
import os, re, glob
from collections import defaultdict

import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import Planck13
import astropy.units as u
from astropy import constants as const

from hyperion.model import ModelOutput

HOME       = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR     = os.path.join(HOME, 'output', 'cis100', 'almac11_specphot')
FIGDIR     = os.path.join(OUTDIR, 'figures')
ALMAC11_CSV = os.path.join(HOME, 'ALMAC11_sed_ism_modeling_results.csv')
OBS_PHOT_HOOK = None   # -> staged CIGALE observations.fits for the ALMA-C11 sample (optional)
AGN_PROP_HOOK = os.path.join(HOME, 'obs_data', 'almac11',
                             'ALMAC11_target+control_jwst_AGN_prop.fits')  # optional
os.makedirs(FIGDIR, exist_ok=True)

ARMS = ('dust_on', 'dust_off', 'nenkova_i90', 'nenkova_i60', 'nenkova_i30')
COL = {'no_AGN': 'tab:blue', 'AGN': 'tab:orange', 'obs': 'k',
       'nenkova_i90': '#b30000', 'nenkova_i60': '#e34a33', 'nenkova_i30': '#fc8d59'}

# ── targets: CSV rows + the two CEERS entries ────────────────────────────────
C = Table.read(ALMAC11_CSV, format='csv')
TARGETS = {}
for r in C:
    tid = str(r['ID']).strip()
    TARGETS[tid] = dict(z=float(r['z']), logm=float(r['log(Mstar/Msol)']),
                        age_gyr=10.0 ** float(r['Age(log/yr)']) / 1e9,
                        av_ism=float(r['bayes.attenuation.Av_ISM']),
                        av_ism_err=float(r['bayes.attenuation.Av_ISM_err']),
                        sample='almac11', row=dict(zip(r.colnames, r)))
TARGETS['719']  = dict(z=1.463, logm=10.29, age_gyr=2.65, av_ism=np.nan, av_ism_err=np.nan,
                       sample='ceers', row=None)
TARGETS['2962'] = dict(z=1.266, logm=10.68, age_gyr=3.30, av_ism=np.nan, av_ism_err=np.nan,
                       sample='ceers', row=None)
_tid_safe = lambda tid: re.sub(r'[^A-Za-z0-9_-]', '', tid)

# ── selection tables + source map ────────────────────────────────────────────
SELECTED = {tid: Table.read(os.path.join(OUTDIR, f'rt_selection_{_tid_safe(tid)}.fits'))
            for tid in TARGETS
            if os.path.exists(os.path.join(OUTDIR, f'rt_selection_{_tid_safe(tid)}.fits'))}
with fits.open(os.path.join(OUTDIR, 'rt_union.fits')) as _h:
    UNION = Table(_h['UNION'].data)
    MEMBERS = Table(_h['MEMBERS'].data)
SMAP = Table.read(os.path.join(OUTDIR, 'almac11_source_map.fits'))

_smap = {}
for r in SMAP:
    if bool(r['COMPLETE']):
        _smap[(str(r['ARM']), int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))] = str(r['RTOUT_PATH'])

def rtout_path(arm, snap, gid):
    return _smap.get((arm, int(snap), int(gid)))   # None if RT not complete

_as_str = lambda col: np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x)
                                for x in col])

# ── SED reader (largest aperture; sightline handling per convention above) ───
_SED_CACHE = {}
def read_sed(arm, snap, gid, sightline='mean'):
    # -> (wav_rest_um, nuLnu_erg_s) or None if the RT is not complete.
    key = (arm, int(snap), int(gid), sightline)
    if key not in _SED_CACHE:
        p = rtout_path(arm, snap, gid)
        if p is None:
            return None
        m = ModelOutput(p)
        sed = m.get_sed(inclination='all', aperture=-1)
        wav = np.asarray(sed.wav, float)
        val = np.atleast_2d(np.asarray(sed.val, float))
        val = val[0] if (sightline == 'findx0' or val.shape[0] == 1) else val.mean(axis=0)
        s = np.argsort(wav)
        _SED_CACHE[key] = (wav[s], val[s])
    return _SED_CACHE[key]

GRID = np.geomspace(0.08, 1500.0, 400)   # rest-frame micron

def on_grid(wl, nuLnu):
    with np.errstate(invalid='ignore', divide='ignore'):
        return 10 ** np.interp(np.log10(GRID), np.log10(wl), np.log10(nuLnu),
                               left=np.nan, right=np.nan)

def obs_fnu_factor(z):
    # (lambda_obs [um], factor nuLnu[erg/s] -> Fnu [mJy]) at redshift z
    wav_obs = GRID * (1.0 + z)
    dl = Planck13.luminosity_distance(z).to(u.cm).value
    nu_obs = (const.c / (wav_obs * u.micron)).to(u.Hz).value
    return wav_obs, 1.0 / (4.0 * np.pi * dl**2 * nu_obs) / 1e-26

print(f'{len(TARGETS)} targets | {len(UNION)} union galaxies | '
      f'{len(SMAP)} source-map rows ({int(np.asarray(SMAP["COMPLETE"]).sum())} complete)')
for arm in ARMS:
    m_ = np.asarray(SMAP['ARM']) == arm
    print(f'  {arm:12s} {int(np.asarray(SMAP["COMPLETE"])[m_].sum()):4d} / {int(m_.sum()):4d} complete')


## Part 1 · observed side — CIGALE best-model fluxes at band pivots

`best.<band>` are model fluxes in mJy (no errors). Pivot wavelengths below are observed-frame
microns; replace with the exact CIGALE filter pivots if a filters directory is staged.


In [ ]:
# ── Part 1 · observed photometry (CIGALE best-model fluxes) ──────────────────
BAND_PIVOT_UM = {              # observed-frame pivot wavelength [um]
    'best.MCam_u': 0.381,  'best.subaru.suprime.IB427': 0.426, 'best.SUBARU_B': 0.446,
    'best.subaru.suprime.IB464': 0.464, 'best.MCam_g': 0.487, 'best.subaru.suprime.V': 0.548,
    'best.subaru.suprime.r': 0.629, 'best.SUBARU_i': 0.768,
    'best.vista.vircam.J': 1.254, 'best.vista.vircam.H': 1.646, 'best.vista.vircam.Ks': 2.149,
    'best.IRAC1': 3.557, 'best.IRAC2': 4.504, 'best.IRAC3': 5.738, 'best.IRAC4': 7.927,
    'best.MIPS1': 23.68, 'best.PACS_green': 100.0, 'best.PSW': 250.0, 'best.ALMA6': 1250.0,
}

OBS = {}
for tid, T in TARGETS.items():
    if T['row'] is None:
        continue                       # CEERS: observed side lives in the cis100 notebook
    wl = np.array([BAND_PIVOT_UM[b] for b in BAND_PIVOT_UM])
    f_ = np.array([float(T['row'][b]) for b in BAND_PIVOT_UM])
    OBS[tid] = dict(wl=wl, fnu=f_)     # mJy, observed frame

# optional: real observed photometry with errors, once staged on the cluster
if OBS_PHOT_HOOK and os.path.exists(OBS_PHOT_HOOK):
    print('observations.fits hook found — wire the per-band errors in here')
print(f'observed side ready for {len(OBS)} ALMA-C11 targets '
      f'({len(BAND_PIVOT_UM)} bands, 0.38 um - 1.25 mm observed)')


## Part 2 · mock stacks per target — dust_on vs observed

Median + 16-84% band of the member analogues' `dust_on` SEDs, placed at the target redshift,
against the CIGALE best-model fluxes. AGN/non-AGN split via the per-target `AGN_CLASS`.


In [ ]:
# ── Part 2 · per-target SED stacks ───────────────────────────────────────────
def stack_curves(rows, arm, sightline='mean'):
    curves = []
    for r in rows:
        sed = read_sed(arm, r['SNAPSHOT'], r['GROUPID_SNAPSHOT'], sightline=sightline)
        if sed is not None:
            curves.append(on_grid(*sed))
    C_ = np.asarray(curves)
    if C_.size == 0:
        return None
    return dict(med=np.nanmedian(C_, axis=0), lo=np.nanpercentile(C_, 16, axis=0),
                hi=np.nanpercentile(C_, 84, axis=0), n=len(C_), curves=C_)

for tid in [t for t in TARGETS if t in SELECTED and t in OBS]:
    A = SELECTED[tid]
    A_cls = _as_str(A['AGN_CLASS'])
    z = TARGETS[tid]['z']
    WOBS, FAC = obs_fnu_factor(z)
    fig, ax = plt.subplots(figsize=(8.5, 6))
    for cls in ('no_AGN', 'AGN'):
        S = stack_curves(A[A_cls == cls], 'dust_on')
        if S is None:
            continue
        ax.fill_between(WOBS, S['lo'] * FAC, S['hi'] * FAC, color=COL[cls], alpha=0.25, lw=0)
        ax.plot(WOBS, S['med'] * FAC, color=COL[cls], lw=2,
                label=f"SIMBA {'AGN host' if cls == 'AGN' else 'non-AGN host'} (N={S['n']})")
    ax.plot(OBS[tid]['wl'], OBS[tid]['fnu'], 'o', ms=6, mfc='w', mec=COL['obs'],
            label='CIGALE best-model fluxes', zorder=5)
    ax.set(xscale='log', yscale='log', xlim=(0.2, 2000),
           xlabel=r'observed-frame wavelength [$\mu$m]', ylabel=r'$F_\nu$ [mJy]',
           title=f"{tid} (z={z:.3f}) vs SIMBA analogues — dust_on")
    fin = OBS[tid]['fnu'][OBS[tid]['fnu'] > 0]
    ax.set_ylim(fin.min() / 3e2, fin.max() * 3e2)
    ax.grid(alpha=0.2, which='both', lw=0.5)
    ax.legend(fontsize=9, frameon=False, loc='lower center')
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, f'sed_stack_{_tid_safe(tid)}.png'),
                dpi=180, bbox_inches='tight')
    plt.show()


## Part 3 · attenuation — A_lambda per galaxy, A_V vs CIGALE Av_ISM

`A_lambda = -2.5 log10(dust_on / dust_off)` per galaxy at matched sightline (`findx=0`);
A_V read at rest 0.55 um. Median attenuation curve per target + the A_V distribution against the
CIGALE `bayes.attenuation.Av_ISM` of that source.


In [ ]:
# ── Part 3 · attenuation curves + A_V comparison ─────────────────────────────
def attenuation_curve(snap, gid):
    on = read_sed('dust_on', snap, gid, sightline='findx0')
    off = read_sed('dust_off', snap, gid, sightline='findx0')
    if on is None or off is None:
        return None
    with np.errstate(invalid='ignore', divide='ignore'):
        return -2.5 * np.log10(on_grid(*on) / on_grid(*off))

_jV = int(np.argmin(np.abs(GRID - 0.55)))
AV = {}          # tid -> array of A_V of its members
fig, ax = plt.subplots(figsize=(8, 5.5))
for tid in [t for t in TARGETS if t in SELECTED]:
    A = SELECTED[tid]
    curves = [attenuation_curve(r['SNAPSHOT'], r['GROUPID_SNAPSHOT']) for r in A]
    curves = np.asarray([c_ for c_ in curves if c_ is not None])
    if curves.size == 0:
        continue
    AV[tid] = curves[:, _jV]
    ax.plot(GRID, np.nanmedian(curves, axis=0), lw=1.2, alpha=0.8, label=tid)
ax.set(xscale='log', xlim=(0.1, 30), xlabel=r'rest-frame wavelength [$\mu$m]',
       ylabel=r'$A_\lambda$ [mag]', title='median attenuation curve per target')
ax.legend(fontsize=7, ncol=3, frameon=False)
ax.grid(alpha=0.2, which='both', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'attenuation_curves.png'), dpi=180)
plt.show()

# A_V distribution vs CIGALE Av_ISM
fig, ax = plt.subplots(figsize=(9, 5))
_tids = [t for t in AV if TARGETS[t]['sample'] == 'almac11']
for x, tid in enumerate(_tids):
    y = AV[tid][np.isfinite(AV[tid])]
    ax.plot(np.full(y.size, x) + np.random.uniform(-0.15, 0.15, y.size), y, 'o', ms=3,
            color='tab:blue', alpha=0.4, mec='none')
    ax.hlines(np.median(y), x - 0.25, x + 0.25, color='tab:blue', lw=2)
    ax.errorbar(x, TARGETS[tid]['av_ism'], yerr=TARGETS[tid]['av_ism_err'], fmt='s', ms=6,
                color='k', capsize=3)
ax.set(xticks=range(len(_tids)), xticklabels=_tids, ylabel=r'$A_V$ [mag]',
       title='simulated analogue $A_V$ (blue) vs CIGALE Av_ISM (black)')
ax.tick_params(axis='x', rotation=60)
ax.grid(alpha=0.2, axis='y', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'av_vs_cigale.png'), dpi=180)
plt.show()


## Part 4 · torus MIR strength vs CLUMPY inclination

AGN hosts only. Per galaxy the torus contribution is the residual `nenkova_iXX − dust_on`
(same galaxy, same dust, only the injected template differs; dust_on at `findx=0`). Median
residuals for i=90/60/30, plus the ratio-to-dust_on version. Optional: rescale each residual by
`L_AGN,obs / L_AGN,injected` (needs the AGN-prop FITS staged — see header).


In [ ]:
# ── Part 4 · torus residual vs inclination ───────────────────────────────────
_agn_rows = UNION[np.asarray(UNION['AGN_ANY'], bool)]
print(f'AGN hosts with torus arms: {len(_agn_rows)}')

RES = {}
for arm in ('nenkova_i90', 'nenkova_i60', 'nenkova_i30'):
    res, ratio = [], []
    for r in _agn_rows:
        snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
        on = read_sed('dust_on', snap, gid, sightline='findx0')
        nk = read_sed(arm, snap, gid, sightline='mean')   # mean handles the reused 4-sightline i90
        if on is None or nk is None:
            continue
        g_on, g_nk = on_grid(*on), on_grid(*nk)
        res.append(g_nk - g_on)
        with np.errstate(invalid='ignore', divide='ignore'):
            ratio.append(g_nk / g_on)
    if res:
        RES[arm] = dict(res=np.asarray(res), ratio=np.asarray(ratio), n=len(res))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for arm, D in RES.items():
    axes[0].plot(GRID, np.nanmedian(D['res'], axis=0), color=COL[arm], lw=2,
                 label=f"{arm} (N={D['n']})")
    axes[1].plot(GRID, np.nanmedian(D['ratio'], axis=0), color=COL[arm], lw=2)
axes[0].set(xscale='log', yscale='log', xlim=(0.1, 500), ylabel=r'$\nu L_\nu$ [erg s$^{-1}$]',
            xlabel=r'rest-frame wavelength [$\mu$m]',
            title='median torus residual (nenkova $-$ dust_on)')
axes[1].axhline(1, color='0.5', lw=0.8)
axes[1].set(xscale='log', yscale='log', xlabel=r'rest-frame wavelength [$\mu$m]',
            ylabel='nenkova / dust_on', title='median boost over the host')
axes[0].legend(fontsize=9, frameon=False)
for ax in axes:
    ax.grid(alpha=0.2, which='both', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'torus_vs_inclination.png'), dpi=180)
plt.show()

# optional L_AGN rescaling: residual * (L_AGN_obs / L_AGN_injected), L_AGN_injected = LBOL (0.1 Mdot c^2)
if os.path.exists(AGN_PROP_HOOK):
    print('AGN-prop FITS found — add the matched-L_AGN panel here')
else:
    print(f'AGN-prop FITS not staged ({AGN_PROP_HOOK}) — matched-L_AGN panel skipped')


## Part 5 · dust masses — sanitized CSV column vs simulated analogues

The CSV `M_dust [Msun]` column has **mixed units across rows** (some raw Msun, some scaled).
Adjudicate row-by-row against `Mdust_sc` (+err) and `Mdust_mbb`; adopt `Mdust_sc` where the raw
column is inconsistent, and print the adjudication so nothing is silent.


In [ ]:
# ── Part 5 · dust-mass comparison ────────────────────────────────────────────
print(f"{'tid':8s} {'M_dust[raw]':>12s} {'Mdust_sc':>12s} {'Mdust_mbb':>12s} {'adopted logMd':>13s}")
LOGMD_OBS = {}
for tid, T in TARGETS.items():
    if T['row'] is None:
        continue
    raw, sc, mbb = (float(T['row'].get(k_, np.nan)) for k_ in
                    ('M_dust [Msun]', 'Mdust_sc', 'Mdust_mbb'))
    # raw is trustworthy only when it agrees with Mdust_sc within a factor of a few
    adopted = raw if (np.isfinite(raw) and np.isfinite(sc) and 0.3 < raw / sc < 3) else sc
    LOGMD_OBS[tid] = np.log10(adopted) if np.isfinite(adopted) and adopted > 0 else np.nan
    flag = '' if adopted is raw else '   <- raw inconsistent, using Mdust_sc'
    print(f'{tid:8s} {raw:12.3e} {sc:12.3e} {mbb:12.3e} {LOGMD_OBS[tid]:13.2f}{flag}')

_tids = [t for t in LOGMD_OBS if t in SELECTED]
fig, ax = plt.subplots(figsize=(9, 5))
for x, tid in enumerate(_tids):
    y = np.asarray(SELECTED[tid]['LOG_MDUST'], float)
    y = y[np.isfinite(y)]
    ax.plot(np.full(y.size, x) + np.random.uniform(-0.15, 0.15, y.size), y, 'o', ms=3,
            color='tab:blue', alpha=0.4, mec='none')
    if y.size:
        ax.hlines(np.median(y), x - 0.25, x + 0.25, color='tab:blue', lw=2)
    ax.plot(x, LOGMD_OBS[tid], 's', ms=7, color='k')
ax.set(xticks=range(len(_tids)), xticklabels=_tids,
       ylabel=r'$\log\,M_{\rm dust}\,[M_\odot]$',
       title='simulated analogue dust masses (blue) vs ALMA-C11 (black)')
ax.tick_params(axis='x', rotation=60)
ax.grid(alpha=0.2, axis='y', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'dust_mass_almac11.png'), dpi=180)
plt.show()
